In [1]:
# Importar librerías
import pandas as pd
from src.config import data_folder
from src.funcionesExtract import *

In [2]:
# Obtener lista de tickers del S&P 500 desde el fichero constituents.csv
# Si no existe el fichero, lo descarga desde GitHub
# Para actualizar la lista de componentes, cambiar force_update a True, ejecutar y volver a dejar en False
ruta_archivo = descargar_constituents(force_update=False) 

# Cargar datos
df_tickers = pd.read_csv(ruta_archivo)
df_tickers.info()

Usando archivo constituents.csv local.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Symbol                 503 non-null    object
 1   Security               503 non-null    object
 2   GICS Sector            503 non-null    object
 3   GICS Sub-Industry      503 non-null    object
 4   Headquarters Location  503 non-null    object
 5   Date added             503 non-null    object
 6   CIK                    503 non-null    int64 
 7   Founded                503 non-null    object
dtypes: int64(1), object(7)
memory usage: 31.6+ KB


In [3]:
# Limpieza previa de df_tickers
df_tickers_clean = limpieza_tickers(df_tickers)

# Lista de tickers
tickers_list = df_tickers_clean["Ticker"].tolist()
df_tickers_clean.head()

,Ticker,Sector,DateAdded
0,MMM,Industrials,1957-03-04
1,AOS,Industrials,2017-07-26
2,ABT,HealthCare,1957-03-04
3,ABBV,HealthCare,2012-12-31
4,ACN,InformationTechnology,2011-07-06


In [4]:
# Extraer datos de SimFin (trimestrales de los últimos 4 años con un lag de 1 año)
df_simfin_raw = extraer_simfin(tickers_list)
df_simfin_raw.info()

Dataset "us-income-quarterly" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-balance-quarterly" on disk (1 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-quarterly" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-income-banks-quarterly" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-balance-banks-quarterly" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-banks-quarterly" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-income-insurance-quarterly" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-balance-insurance-quarterly" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-insurance-quarterly" on disk (0 days old).
- Loading from disk ... Done!
<class 'pandas.core.frame.DataFrame'>
Index: 6692 entries, 0 to 50229
Data columns (total 82 columns):
 #   Column                                           Non-Null Count  Dtype         
---  ------      

In [5]:
# Identificar los tickers que SimFin devolvió exitosamente
tickers_simfin = df_simfin_raw['Ticker'].unique().tolist()
print(f"Tickers obtenidos de SimFin: {len(tickers_simfin)}")
print(f"Tickers descartados por limitación: {len(tickers_list) - len(tickers_simfin)}")

Tickers obtenidos de SimFin: 384
Tickers descartados por limitación: 119


In [6]:
# Se acota el Universo de tickers, limitado por aquellos que tienen datos disponibles en la versión gratuita de SimFin
# Se guarda el fichero del universo de tickers restringido
tickers_universe = pd.DataFrame(tickers_simfin, columns=["Ticker"])
tickers_universe.to_csv(f'{data_folder}/tickers_universe.csv')

In [ ]:
# Extraer precios de los tickers y del índice SPY (demora unos 2 minutos)
df_prices = extraer_precios(tickers_simfin)

# Se extrae precio del Índice de Mercado para usar en cálculos y se guarda en fichero
df_index = extraer_precios(['SPY'])
df_index.to_parquet(f"{data_folder}/market_index.parquet")

df_prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27501 entries, 0 to 27500
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       27501 non-null  datetime64[ns]
 1   Close      27501 non-null  float64       
 2   Dividends  27501 non-null  float64       
 3   Ticker     27501 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(1)
memory usage: 859.5+ KB


In [8]:
df_index.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       72 non-null     datetime64[ns]
 1   Close      72 non-null     float64       
 2   Dividends  72 non-null     float64       
 3   Ticker     72 non-null     object        
dtypes: datetime64[ns](1), float64(2), object(1)
memory usage: 2.4+ KB


In [9]:
# Unir df_prices y df_tickers
df_merged = pd.merge(
    df_prices,
    df_tickers_clean,
    on= 'Ticker',
    how= 'left'
)
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27501 entries, 0 to 27500
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       27501 non-null  datetime64[ns]
 1   Close      27501 non-null  float64       
 2   Dividends  27501 non-null  float64       
 3   Ticker     27501 non-null  object        
 4   Sector     27501 non-null  object        
 5   DateAdded  27501 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(3)
memory usage: 1.3+ MB


In [ ]:
# Extraer datos financieros de yfinance (ultimos 4 reportes trimestrales, demora unos 8 minutos)
df_yfinance = extraer_financials(tickers_simfin)
df_yfinance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1536 entries, 0 to 1535
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Date                       1536 non-null   datetime64[ns]
 1   Total Revenue              1532 non-null   float64       
 2   Gross Profit               1425 non-null   float64       
 3   Operating Income           1436 non-null   float64       
 4   Net Income                 1532 non-null   float64       
 5   EBITDA                     1433 non-null   float64       
 6   Basic Average Shares       1506 non-null   float64       
 7   Cash And Cash Equivalents  1530 non-null   float64       
 8   Current Debt               1144 non-null   float64       
 9   Long Term Debt             1417 non-null   float64       
 10  Total Debt                 1506 non-null   float64       
 11  Stockholders Equity        1527 non-null   float64       
 12  Total 

In [11]:
# Definir columnas a mantener en simfin para que coincidan y estandarizar antes de unir
cols_yfinance = df_yfinance.columns
df_simfin_clean = estandarizar_simfin(df_simfin_raw, cols_yfinance)

df_financials_completo = unir_financials(df_yfinance, df_simfin_clean)
df_financials_completo.info()

Se han encontrado 0 filas con Ticker y Date solapados.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8212 entries, 0 to 8211
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Ticker                     8212 non-null   object        
 1   Date                       8203 non-null   datetime64[ns]
 2   Total Revenue              8199 non-null   float64       
 3   Gross Profit               7388 non-null   float64       
 4   Operating Income           8103 non-null   float64       
 5   Net Income                 8199 non-null   float64       
 6   EBITDA                     4504 non-null   float64       
 7   Basic Average Shares       8168 non-null   float64       
 8   Cash And Cash Equivalents  8176 non-null   float64       
 9   Current Debt               6094 non-null   float64       
 10  Long Term Debt             7619 non-null   float64       
 11  Total Debt    

In [12]:
# Unir dataframe de precios con datos financieros
df_final = pd.merge(
    df_merged, 
    df_financials_completo, 
    on=['Date', 'Ticker'],
    how='left'
)
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27501 entries, 0 to 27500
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Date                       27501 non-null  datetime64[ns]
 1   Close                      27501 non-null  float64       
 2   Dividends                  27501 non-null  float64       
 3   Ticker                     27501 non-null  object        
 4   Sector                     27501 non-null  object        
 5   DateAdded                  27501 non-null  object        
 6   Total Revenue              8191 non-null   float64       
 7   Gross Profit               7381 non-null   float64       
 8   Operating Income           8095 non-null   float64       
 9   Net Income                 8191 non-null   float64       
 10  EBITDA                     4499 non-null   float64       
 11  Basic Average Shares       8158 non-null   float64       
 12  Cash

In [13]:
# Limpieza final, con forward fill para datos financieros (se llenan datos mensuales con el último dato trimestral)
df_final_clean = limpieza_final(df_final)
df_final_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13566 entries, 0 to 13565
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Date                    13566 non-null  datetime64[ns]
 1   Close                   13566 non-null  float64       
 2   Dividends               13566 non-null  float64       
 3   Ticker                  13566 non-null  object        
 4   Sector                  13566 non-null  object        
 5   DateAdded               13566 non-null  object        
 6   TotalRevenue            13566 non-null  float64       
 7   GrossProfit             13087 non-null  float64       
 8   OperatingIncome         13566 non-null  float64       
 9   NetIncome               13566 non-null  float64       
 10  EBITDA                  13566 non-null  float64       
 11  BasicAverageShares      13561 non-null  float64       
 12  CashAndCashEquivalents  13557 non-null  float6

In [14]:
# Guardar datos extraidos en fichero raw_data
df_final_clean.to_parquet(f"{data_folder}/raw_data.parquet")
print("Fichero guardado en la carpeta",data_folder)

Fichero guardado en la carpeta data
